# Diffusion-Limited Aggregation (DLA)
## Capstone Project Notebook

Diffusion-Limited Aggregation (DLA) is a process in which particles undergoing a random walk (diffusion) cluster together to form aggregates. The process was first described by Witten and Sander in 1981 and has since become one of the most studied models of pattern formation in physics.

The key insight of DLA is that **extremely simple rules** — random motion plus irreversible sticking — produce structures of remarkable **complexity and beauty**. DLA clusters are **fractal**: they exhibit self-similarity across scales, with a fractal dimension of approximately **1.71** in two dimensions. This means they are more space-filling than a line ($D = 1$) but far less dense than a filled disc ($D = 2$).

### Where Does DLA Appear?

DLA-like patterns are ubiquitous in nature and technology:

- **Electrochemical deposition** — metals deposited from solution onto an electrode form branching, tree-like structures
- **Mineral dendrites** — manganese oxide deposits in limestone cracks resemble DLA clusters
- **Lightning and dielectric breakdown** — electrical discharge paths through air follow DLA-like branching
- **Bacterial colony growth** — under nutrient-limited conditions, bacterial colonies grow in fractal patterns
- **Snowflake formation** — ice crystal growth shares features with diffusion-limited processes
- **Viscous fingering** — when a less viscous fluid displaces a more viscous one (e.g., water into oil)

### The DLA Algorithm

The algorithm is deceptively simple:

1. Place a **seed particle** at the centre of a grid
2. Release a new particle far from the existing cluster
3. The particle performs a **random walk** until it either:
   - Lands **adjacent** to an occupied cell → it **sticks** and becomes part of the cluster
   - Wanders too far away → discard and try again
4. Repeat for the desired number of particles

The resulting cluster has a characteristic branching morphology: outer tips grow fastest because they intercept diffusing particles first, while inner regions are "screened" from incoming particles. This **tip-splitting instability** is the fundamental mechanism behind DLA's fractal structure.

### In this notebook you will:
1. Understand how **random walks** and **sticking rules** produce fractal patterns
2. Implement a basic **DLA simulation** step by step
3. Visualise the growth of fractal-like aggregates
4. Measure the **fractal dimension** using box-counting
5. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for grid operations and random number generation, and Matplotlib for visualising the fractal clusters. DLA simulations are computationally intensive (each particle may take thousands of random-walk steps), so efficient array operations are important.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## 1 · What is DLA?

Diffusion-Limited Aggregation (Witten & Sander, 1981) is a simple growth model:

1. Place a **seed particle** at the centre of a grid
2. Release a new particle far from the cluster
3. The particle performs a **random walk** until it:
   - **Sticks** to a neighbour of the existing cluster, or
   - Wanders too far away (discard and try again)
4. Repeat

Despite these trivial rules, DLA produces remarkably complex, **fractal** structures that resemble:
- Electrochemical deposits
- Lightning bolts (Lichtenberg figures)
- Bacterial colony growth
- Mineral dendrites in rocks

The fractal dimension of 2D DLA clusters is approximately **1.71**.

### Step 1: Check adjacency

The sticking rule in DLA is simple: a diffusing particle joins the cluster when it lands on a cell that is **adjacent** to an already-occupied cell. Here we use 4-connectivity (von Neumann neighbourhood — up, down, left, right) rather than 8-connectivity (Moore neighbourhood). This choice affects cluster morphology: 4-connected sticking produces slightly more open, branching structures because diagonal approaches don't count as contact.

The function returns `True` if any of the four cardinal neighbours of cell $(r, c)$ is occupied, and `False` otherwise. We test it with a single seed particle at the grid centre.

In [ ]:
def is_adjacent_to_cluster(grid, r, c):
    """Check if cell (r,c) is adjacent to an occupied cell (4-connected)."""
    rows, cols = grid.shape
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 1:
            return True
    return False

### Step 2: Random walk of a single particle

Each new particle is spawned on a circle surrounding the current cluster and then performs a **random walk** — stepping one cell in a random cardinal direction at each iteration. The walk continues until one of two things happens:

- The particle lands **adjacent to the cluster** → it **sticks** (irreversibly joins the cluster). This is the growth event.
- The particle wanders beyond the **kill radius** → it is **discarded** and a new particle is launched. This prevents wasting computation on particles that have diffused far from the cluster.

The spawn radius is set slightly larger than the current cluster radius so particles approach from all directions equally. The kill radius is set much larger to give particles a fair chance of reaching the cluster. The maximum walk length (50,000 steps) is a safety limit to prevent infinite loops.

This is the most computationally expensive part of DLA: most of a particle's walk is spent diffusing through empty space far from the cluster. Advanced implementations use techniques like "circle-jumping" to skip over empty regions.

In [ ]:
def walk_particle(grid, spawn_radius, kill_radius, rng):
    """Launch one particle, random-walk it until it sticks or escapes.
    
    Returns (r, c) of sticking position, or None if particle escaped.
    """
    rows, cols = grid.shape
    centre = rows // 2
    # Spawn on a circle of given radius
    angle = rng.random() * 2 * np.pi
    r = int(centre + spawn_radius * np.sin(angle))
    c = int(centre + spawn_radius * np.cos(angle))
    r = max(0, min(rows-1, r))
    c = max(0, min(cols-1, c))
    
    for _ in range(50000):  # max walk steps
        # Check stick
        if is_adjacent_to_cluster(grid, r, c):
            return r, c
        # Move
        direction = rng.randint(0, 3)
        if direction == 0: r -= 1
        elif direction == 1: r += 1
        elif direction == 2: c -= 1
        else: c += 1
        # Bounds
        dist = np.sqrt((r - centre)**2 + (c - centre)**2)
        if dist > kill_radius or r < 0 or r >= rows or c < 0 or c >= cols:
            return None  # escaped
    return None

### Step 3: Full DLA simulation

The `simulate_dla` function orchestrates the entire growth process. It starts with a single seed particle at the grid centre, then launches particles one at a time. Each particle random-walks until it sticks or escapes. The function dynamically adjusts the spawn and kill radii as the cluster grows, keeping them proportional to the current cluster extent.

Progress is printed every 200 particles because DLA can be slow — 800 particles may require millions of random-walk steps in total. The function returns the final grid and a list of cluster sizes after each particle launch, which we can use to analyse the growth rate.

In [ ]:
def simulate_dla(grid_size=201, n_particles=800, seed=42):
    """Run a DLA simulation. Returns the grid and cluster size over time."""
    rng = np.random.default_rng(seed)
    grid = np.zeros((grid_size, grid_size), dtype=int)
    centre = grid_size // 2
    grid[centre, centre] = 1  # seed
    
    max_r = 5  # current max radius of cluster
    sizes = [1]
    
    for i in range(n_particles):
        spawn_r = max_r + 10
        kill_r = max_r + 50
        result = walk_particle(grid, spawn_r, kill_r, rng)
        if result is not None:
            r, c = result
            grid[r, c] = 1
            dist = np.sqrt((r - centre)**2 + (c - centre)**2)
            if dist > max_r:
                max_r = dist
        sizes.append(int(np.sum(grid)))
    
    return grid, sizes

grid_dla, sizes = simulate_dla(grid_size=201, n_particles=800)

### Animated cluster growth

Static images show the final cluster, but an animation reveals the growth dynamics — how outer tips extend preferentially, how screening prevents inner growth, and how the branching structure emerges particle by particle. Below we re-run a smaller DLA simulation (400 particles) saving snapshots every 5 particles, then render the growth as an animation.

In [ ]:
# --- Pre-compute DLA growth snapshots ---
def simulate_dla_animated(grid_size=151, n_particles=400, snapshot_every=5, seed=42):
    """Run DLA and save grid snapshots at regular intervals."""
    rng = np.random.default_rng(seed)
    grid = np.zeros((grid_size, grid_size), dtype=int)
    centre = grid_size // 2
    grid[centre, centre] = 1
    max_r = 5
    snapshots = [grid.copy()]

    for i in range(n_particles):
        spawn_r = max_r + 10
        kill_r = max_r + 50
        result = walk_particle(grid, spawn_r, kill_r, rng)
        if result is not None:
            r, c = result
            grid[r, c] = 1
            dist = np.sqrt((r - centre)**2 + (c - centre)**2)
            if dist > max_r:
                max_r = dist
        if (i + 1) % snapshot_every == 0:
            snapshots.append(grid.copy())
    return snapshots

dla_snapshots = simulate_dla_animated(grid_size=151, n_particles=400, snapshot_every=5)

# --- Animation ---
fig, ax = plt.subplots(figsize=(6, 6))
image_display_handle = ax.imshow(dla_snapshots[0], cmap='hot', interpolation='nearest')
ax.axis('off')

def update_plot(frame_number):
    """Update the image with the next DLA growth snapshot."""
    image_display_handle.set_data(dla_snapshots[frame_number])
    n_particles = np.sum(dla_snapshots[frame_number])
    ax.set_title(f'DLA Growth — {n_particles} particles')
    return [image_display_handle]

plt.close()

dla_animation = animation.FuncAnimation(
    fig, update_plot, frames=len(dla_snapshots), interval=100, blit=True
)

HTML(dla_animation.to_jshtml())

---
## 2 · Visualise the Aggregate

Two views reveal different aspects of the DLA cluster:

- **Spatial image** (left) — shows the fractal morphology directly. Notice the characteristic branching: outer tips are prominent while the interior is relatively sparse. This is because diffusing particles are more likely to encounter (and stick to) the outermost tips before penetrating deeper — a phenomenon called **screening** or **shadowing**.
- **Growth curve** (right) — plots cluster size vs number of particles launched. Not every launch results in a successful sticking event (some particles escape), so the curve may be slightly sub-linear. The growth rate slows as the cluster gets larger because the spawn radius increases, making walks longer.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.imshow(grid_dla, cmap='hot', interpolation='nearest')
ax1.set_title('DLA Cluster', fontweight='bold')
ax1.axis('off')

ax2.plot(sizes, lw=2, color='#e74c3c')
ax2.set_xlabel('Particles launched')
ax2.set_ylabel('Cluster size')
ax2.set_title('Cluster Growth', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 3 · Estimating Fractal Dimension

A key quantitative property of DLA clusters is their **fractal dimension** $D$. Unlike regular geometric shapes (where dimension is an integer — 1 for lines, 2 for surfaces), fractals have non-integer dimensions that quantify how they fill space.

The **box-counting** method is the standard approach: cover the cluster with a grid of square boxes of side length $\epsilon$ and count how many boxes $N(\epsilon)$ contain at least one particle. For a fractal object:

$$N(\epsilon) \propto \epsilon^{-D}$$

Taking logarithms: $\log N = -D \log \epsilon + \text{const}$, so a plot of $\log N$ vs $\log(1/\epsilon)$ should be linear with slope $D$.

For 2D DLA clusters, the theoretical fractal dimension is approximately **1.71** — significantly less than 2 (a solid disc) because of the branching, porous structure. We estimate $D$ by fitting a line through the log-log data for several box sizes.

In [ ]:
def box_count(grid, box_sizes):
    counts = []
    for bs in box_sizes:
        n_boxes = 0
        for r in range(0, grid.shape[0], bs):
            for c in range(0, grid.shape[1], bs):
                if np.any(grid[r:r+bs, c:c+bs] > 0):
                    n_boxes += 1
        counts.append(n_boxes)
    return np.array(counts)

box_sizes = [2, 4, 8, 16, 32, 64]
counts = box_count(grid_dla, box_sizes)

log_inv_eps = np.log(1.0 / np.array(box_sizes))
log_N = np.log(counts)
coeffs = np.polyfit(log_inv_eps, log_N, 1)
D_est = coeffs[0]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(log_inv_eps, log_N, s=60, zorder=3)
ax.plot(log_inv_eps, np.polyval(coeffs, log_inv_eps), 'r--', lw=2,
        label=f'Fit: D = {D_est:.2f}')
ax.set_xlabel('$\\log(1/\\epsilon)$')
ax.set_ylabel('$\\log N(\\epsilon)$')
ax.set_title('Box-Counting Fractal Dimension', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4 · Your Tasks

Implement the basic DLA (done above) and add **at least two** features:

### Task A: Seed Growth Variation
Try different seeds: single point, horizontal line, circle, multiple points. How does seed shape affect cluster morphology?

### Task B: Adhesion Probability
Instead of always sticking on contact, stick with probability $p$. Lower $p$ → denser, more compact clusters. Measure D vs $p$.

### Task C: Directional Bias
Bias the random walk (e.g., downward for gravity). Show how bias changes the cluster shape.

### Task D: Obstacle Interaction
Place barriers in the grid and observe how diffusion patterns adapt.

### Task E: Multi-Species Aggregation
Two particle types with different sticking rules — one sticks to anything, another only to its own kind.

In [ ]:
# TODO: Delete this cell

---
## Recommended Reading & Journal Club

**1. Witten, T. A. & Sander, L. M. (1981)** *Diffusion-limited aggregation, a kinetic critical phenomenon.* Physical Review Letters, 47(19), 1400. [DOI](https://doi.org/10.1103/PhysRevLett.47.1400)
→ The original DLA paper.

**2. Meakin, P. (1998)** *Fractals, Scaling and Growth Far from Equilibrium.* Cambridge University Press.
→ Comprehensive treatment of DLA and related growth models.

**3. Vicsek, T. (1992)** *Fractal Growth Phenomena.* 2nd ed. World Scientific.
→ Classic textbook on fractal growth including DLA.

**4. Sander, L. M. (2000)** *Diffusion-limited aggregation: A kinetic critical phenomenon?* Contemporary Physics, 41(4), 203–218. [DOI](https://doi.org/10.1080/001075100409698)
→ Modern review of DLA theory and open questions.